In [12]:
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [14]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

model_path = "C:/Users/sofie/Desktop/ENSAE/3A/S2/NLP/Repo/Detect-LLM/outputs/distilbert_model"  

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

trainer = Trainer(
    model=model,
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 11108.76it/s]


In [15]:
print(type(model))
print(model.config.num_labels)

<class 'transformers.models.distilbert.modeling_distilbert.DistilBertForSequenceClassification'>
2


In [32]:
def prepare_bert_dataset(df, text_col="text", label_col="generated"):
    df = df[[text_col, label_col]].copy()
    df[text_col] = df[text_col].astype(str)
    df[label_col] = df[label_col].astype(int)

    dataset = Dataset.from_pandas(df)
    dataset = dataset.rename_column(label_col, "labels")

    def tokenize_function(batch):
        return tokenizer(
            batch[text_col],
            padding="max_length",
            truncation=True,
            max_length=512
        )

    dataset = dataset.map(tokenize_function, batched=True)

    dataset.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    return dataset


def evaluate_bert_on_dataset(df, dataset_name, threshold=0.5, text_col="text", label_col="generated"):
    dataset = prepare_bert_dataset(df, text_col=text_col, label_col=label_col)

    predictions = trainer.predict(dataset)

    logits = predictions.predictions
    y_true = predictions.label_ids

    probs = softmax(logits, axis=1)
    ai_probs = probs[:, 1]

    y_pred = (ai_probs >= threshold).astype(int)

    print("=" * 80)
    print(f"RESULTS ON: {dataset_name}")
    print(f"AI threshold: {threshold:.2f}")
    print("=" * 80)

    print("Accuracy :", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, zero_division=0))
    print("Recall   :", recall_score(y_true, y_pred, zero_division=0))
    print("F1 score :", f1_score(y_true, y_pred, zero_division=0))

    print("\nClassification report:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=["Human", "AI"],
        zero_division=0
    ))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_true, y_pred))

    return y_true, y_pred, ai_probs

In [30]:
from scipy.special import softmax
from sklearn.metrics import f1_score, precision_score, recall_score

validation_df = pd.read_csv(
    r"C:/Users/sofie/Desktop/ENSAE/3A/S2/NLP/Repo/Detect-LLM/clean_data/Kaggle/kaggle_validation.csv"
)

validation_dataset = prepare_bert_dataset(
    validation_df,
    text_col="text",
    label_col="generated"
)

val_predictions = trainer.predict(validation_dataset)

val_logits = val_predictions.predictions
y_val_true = val_predictions.label_ids

val_probs = softmax(val_logits, axis=1)
val_ai_probs = val_probs[:, 1]

thresholds = np.arange(0.1, 0.91, 0.01)

results = []

for t in thresholds:
    y_val_pred = (val_ai_probs >= t).astype(int)

    results.append({
        "threshold": t,
        "f1": f1_score(y_val_true, y_val_pred),
        "precision": precision_score(y_val_true, y_val_pred, zero_division=0),
        "recall": recall_score(y_val_true, y_val_pred, zero_division=0)
    })

threshold_results = pd.DataFrame(results)
threshold_results.sort_values("f1", ascending=False).head(10)

Map: 100%|██████████| 5258/5258 [00:03<00:00, 1385.73 examples/s]
C:\Users\sofie\miniconda3\envs\NLP_project\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 658/658 [07:57<00:00,  1.38it/s]


,threshold,f1,precision,recall
75,0.85,0.996897,0.996183,0.997611
80,0.90,0.996897,0.996183,0.997611
79,0.89,0.996897,0.996183,0.997611
76,0.86,0.996897,0.996183,0.997611
74,0.84,0.996897,0.996183,0.997611
77,0.87,0.996897,0.996183,0.997611
78,0.88,0.996897,0.996183,0.997611
65,0.75,0.996659,0.995708,0.997611
73,0.83,0.996659,0.995708,0.997611
72,0.82,0.996659,0.995708,0.997611


In [31]:
best_threshold = threshold_results.sort_values("f1", ascending=False).iloc[0]["threshold"]
best_threshold

np.float64(0.8499999999999996)

In [33]:
kaggle_test = pd.read_csv("C:/Users/sofie/Desktop/ENSAE/3A/S2/NLP/Repo/Detect-LLM/clean_data/Kaggle/kaggle_test.csv")

y_true_kaggle, y_pred_kaggle, ai_probs_kaggle = evaluate_bert_on_dataset(
    kaggle_test,
    dataset_name="Kaggle test set",
    threshold=best_threshold
)

Map: 100%|██████████| 5258/5258 [00:03<00:00, 1375.86 examples/s]
C:\Users\sofie\miniconda3\envs\NLP_project\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 658/658 [09:45<00:00,  1.12it/s]


RESULTS ON: Kaggle test set
AI threshold: 0.85
Accuracy : 0.9961962723469
Precision: 0.9933365064255116
Recall   : 0.9971333014811276
F1 score : 0.9952312827849309

Classification report:
              precision    recall  f1-score   support

       Human       1.00      1.00      1.00      3165
          AI       0.99      1.00      1.00      2093

    accuracy                           1.00      5258
   macro avg       1.00      1.00      1.00      5258
weighted avg       1.00      1.00      1.00      5258


Confusion matrix:
[[3151   14]
 [   6 2087]]


In [34]:
medium_data = pd.read_csv("C:/Users/sofie/Desktop/ENSAE/3A/S2/NLP/Repo/Detect-LLM/raw_data/Medium_data.csv")

human_sources = {"guardian", "arxiv"}

medium_data["generated"] = (~medium_data["source"].isin(human_sources)).astype(int)

y_true_medium, y_pred_medium, ai_probs_medium = evaluate_bert_on_dataset(
    medium_data,
    dataset_name="Medium external dataset",
    threshold=best_threshold
)

Map: 100%|██████████| 5000/5000 [00:04<00:00, 1072.33 examples/s]
C:\Users\sofie\miniconda3\envs\NLP_project\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 625/625 [08:37<00:00,  1.21it/s]

RESULTS ON: Medium external dataset
AI threshold: 0.85
Accuracy : 0.851
Precision: 0.838903743315508
Recall   : 0.9941906522313176
F1 score : 0.9099697885196375

Classification report:
              precision    recall  f1-score   support

       Human       0.96      0.40      0.57      1213
          AI       0.84      0.99      0.91      3787

    accuracy                           0.85      5000
   macro avg       0.90      0.70      0.74      5000
weighted avg       0.87      0.85      0.83      5000


Confusion matrix:
[[ 490  723]
 [  22 3765]]
